In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn import metrics
import rasterio as rio
import numpy as np
import os
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
from glob import glob
import json

from mslandcover.data import utils
from mslandcover.utils import get_torch_device
from mslandcover.config import LEGEND_CLASSES
import mslandcover.data.transforms as T


from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple, Optional


In [ ]:
class TestDataset(Dataset):
    def __init__(self, points_gdf: gpd.GeoDataFrame, raster_paths: List[str], n_bands: int = 3, mean: Optional[torch.Tensor] = None, std: Optional[torch.Tensor] = None):
        self.raster_paths = raster_paths
        self.n_bands = n_bands
        self.mean = mean
        self.std = std
        
        self.points_gdf = points_gdf.loc[points_gdf['ground_truth'] != 0]
    
    def __len__(self):
        return len(self.points_gdf)
    
    def __getitem__(self, idx):
        
        point = self.points_gdf.iloc[idx]
        
        raster_id = point['id']
        raster_path = [path for path in self.raster_paths if raster_id + '.tif' == os.path.basename(path)]
        
        assert len(raster_path) == 1, f"Raster path not found for point index {idx}"
        raster_path = raster_path[0]
        
        img, meta = utils.read_image(raster_path, as_float=True, as_tensor=True, return_metadata=True)
        
        if self.n_bands == 3:
            nir_band = img[3, :, :]
            red_band = img[0, :, :]
            green_band = img[1, :, :]
            img = torch.stack([red_band, green_band, nir_band], dim=0)
        
        if self.mean is not None and self.std is not None:
            img = T.normalize(img, mean=self.mean, std=self.std)
        
        x, y = point.geometry.x, point.geometry.y
        # row, col = meta['transform'].index(x, y, op='round')
        row, col = rio.transform.rowcol(meta['transform'], x, y)
        row, col = int(row), int(col)
        
        class_idx = point['ground_truth']
        class_name = point['ground_truth_class_name']
        
        return_dict = {
            'image': img,
            'row': row,
            'col': col,
            'class_idx': class_idx,
            'class_name': class_name,
            'point_id': point['id'],
        }
        for key, value in return_dict.items():
            if value is None:
                raise ValueError(f"Missing value for key: {key} in point index {idx}")
        

In [33]:
samples_gdf = gpd.read_file(r"D:\new_mslc_data\assessment\assessment_points")
samples_gdf = samples_gdf.rename(columns={'ground_tru': 'ground_truth', 'ground_t_1': 'ground_truth_class_name'})
samples_gdf = samples_gdf[samples_gdf['ground_truth'] != 0]
display(samples_gdf.head())
print(f"Number of samples: {len(samples_gdf)}")

,lat,lon,id,ground_truth,ground_truth_class_name,GlobalID,geometry
0,32.424252,-89.940370,0000,5,Tree Canopy/Woody Vegetation,{3C35228C-E839-4CE4-93C5-DD505D341001},POINT (482097.672 1291617.241)
1,33.691628,-90.771233,0001,7,Cultivated Crops,{F5A1C2E3-B1EC-419D-BD4E-EEBC120DAC26},POINT (405327.857 1432604.97)
2,33.705656,-88.989263,0002,4,Barren Land,{8F25EEBF-5E96-4219-BE84-CF0CAD91CDE6},POINT (570511.086 1433952.39)
3,31.388136,-90.781755,0003,5,Tree Canopy/Woody Vegetation,{25AD45DA-2697-4BDC-99D9-E0731A76CF4D},POINT (401878.252 1177190.675)
4,34.734996,-90.026508,0004,5,Tree Canopy/Woody Vegetation,{38713756-4C42-4A77-984D-43CB813D5C01},POINT (474681.164 1547889.368)


Number of samples: 10000


In [22]:
test_dataset = TestDataset(
    points_gdf=samples_gdf.iloc[::-1],  # Reverse the order of the DataFrame
    raster_paths=glob(r"D:\new_mslc_data\assessment\reduced_res\*.tif"),
    n_bands=3,
    mean=torch.load('./weights/pretrain_mean.pth'),
    std=torch.load('./weights/pretrain_std.pth')
)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_38880\1942463231.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mean=torch.load('./weights/pretrain_mean.pth'),
C:\Users\dh

In [23]:
from mslandcover.models import UNet, ResNetBackboneUNet

In [24]:
model = UNet(
    backbone=ResNetBackboneUNet(in_channels=3, pretrained=False),
    num_classes=8,
)

model.load_state_dict(torch.load('./weights/finetune_20250607/unet_fe.pth'))
model.eval()
torch.set_grad_enabled(False)

device = get_torch_device()
model.to(device)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_38880\2828045811.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./weights/finetune_202506

UNet(
  (backbone): ResNetBackboneUNet(
    (initial): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
       

In [25]:
preds_dict = {
    'point_id': [],
    'predicted_class_idx': [],
    'ground_truth_class_idx': [],
    'ground_truth_class_name': [],
    'cross_entropy': [],
    'brier_score': []
}

test_loader = DataLoader(test_dataset, batch_size=16, pin_memory=False, shuffle=False)

with torch.no_grad():
    
    for batch in tqdm(test_loader):
        continue
        # images = batch['image']
        # images = images.to(device)
        # outputs = model(images)
        
        # preds = torch.argmax(outputs, dim=1)
        # for i in range(len(preds)):
        #     point_id = batch['point_id'][i]
        #     ground_truth_class_idx = batch['class_idx'][i].item()
        #     ground_truth_class_name = batch['class_name'][i]
            
        #     pred_idx = preds[i][batch['row'][i], batch['col'][i]].item()
        #     # hacky way to calculate cross-entropy loss for the specific pixel
        #     cross_entropy = F.cross_entropy(outputs[i][:, batch['row'][i], batch['col'][i]].unsqueeze(0).to('cpu'), torch.tensor([ground_truth_class_idx - 1])).item()
        #     brier_score = F.mse_loss(outputs[i][:, batch['row'][i], batch['col'][i]].unsqueeze(0).to('cpu'), torch.nn.functional.one_hot(torch.tensor([ground_truth_class_idx - 1]), num_classes=8).float()).item()
        #     pred_idx = pred_idx + 1  # Adjusting for zero-based index after calculation
            
        #     preds_dict['point_id'].append(point_id)
        #     preds_dict['predicted_class_idx'].append(pred_idx)
        #     preds_dict['ground_truth_class_idx'].append(ground_truth_class_idx)
        #     preds_dict['ground_truth_class_name'].append(ground_truth_class_name)
        #     preds_dict['cross_entropy'].append(cross_entropy)
        #     preds_dict['brier_score'].append(brier_score)

  0%|          | 0/625 [00:00<?, ?it/s]


ValueError: Missing value for key: class_name in point index 0

In [ ]:
preds_df = pd.DataFrame(preds_dict)

class_counts = preds_df['ground_truth_class_idx'].value_counts(normalize=True).sort_index()
weighted_cross_entropy = (
    preds_df.groupby('ground_truth_class_idx')['cross_entropy'].mean() * class_counts
).sum()
weighted_brier_score = (
    preds_df.groupby('ground_truth_class_idx')['brier_score'].mean() * class_counts
).sum()

metrics_dict = {
    # overall metrics
    'accuracy': metrics.accuracy_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx']),
    'f1_score': metrics.f1_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='micro'),
    'precision': metrics.precision_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='micro'),
    'recall': metrics.recall_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='micro'),
    'jaccard': metrics.jaccard_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='micro'),
    'kappa': metrics.cohen_kappa_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx']),
    'cross_entropy': np.mean(preds_df['cross_entropy']),
    'brier_score': np.mean(preds_df['brier_score']),
    # macro metrics
    'macro_f1_score': metrics.f1_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='macro'),
    'macro_precision': metrics.precision_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='macro'),
    'macro_recall': metrics.recall_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='macro'),
    'macro_jaccard': metrics.jaccard_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='macro'),
    'macro_cross_entropy': np.mean(preds_df.groupby('ground_truth_class_idx')['cross_entropy'].mean()),
    'macro_brier_score': np.mean(preds_df.groupby('ground_truth_class_idx')['brier_score'].mean()),
    # weighted metrics
    'weighted_f1_score': metrics.f1_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='weighted'),
    'weighted_precision': metrics.precision_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='weighted'),
    'weighted_recall': metrics.recall_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='weighted'),
    'weighted_jaccard': metrics.jaccard_score(preds_df['ground_truth_class_idx'], preds_df['predicted_class_idx'], average='weighted'),
    'weighted_cross_entropy': weighted_cross_entropy,
    'weighted_brier_score': weighted_brier_score,
}
print(json.dumps(metrics_dict, indent=4))


{
    "accuracy": 0.921875,
    "f1_score": 0.921875,
    "precision": 0.921875,
    "recall": 0.921875,
    "jaccard": 0.855072463768116,
    "macro_f1_score": 0.8645762958119938,
    "cross_entropy": 1.6864769030362368,
    "macro_precision": 0.9199204914403778,
    "macro_recall": 0.8465668202764978,
    "macro_jaccard": 0.7788048552754435,
    "macro_cross_entropy": 1.664577039089483,
    "weighted_f1_score": 0.9218182457956486,
    "weighted_precision": 0.9342515941681425,
    "weighted_recall": 0.921875,
    "weighted_jaccard": 0.8643552559912854,
    "weighted_cross_entropy": 1.686476903036237,
    "kappa": 0.8605934459557578
}


In [ ]:
classification_report = metrics.classification_report(
    preds_df['ground_truth_class_idx'],
    preds_df['predicted_class_idx'],
    target_names=[LEGEND_CLASSES[i] for i in range(1, 9)],
    output_dict=True
)

classification_report_df = pd.DataFrame(classification_report).transpose()
classification_report_df

,precision,recall,f1-score,support
Open Water,0.909091,1.000000,0.952381,20.000
Impervious Structures,0.600000,0.600000,0.600000,5.000
Impervious Surfaces,0.294118,0.714286,0.416667,7.000
Barren Land,0.944444,0.607143,0.739130,84.000
Tree Canopy/Woody Vegetation,0.933977,0.957096,0.945395,606.000
Low Vegetation/Herbaceous,0.742857,0.778443,0.760234,167.000
Cultivated Crops,0.786667,0.967213,0.867647,61.000
Unclassified,0.741935,0.460000,0.567901,50.000
accuracy,0.871000,0.871000,0.871000,0.871
macro avg,0.744136,0.760523,0.731169,1000.000
